In [87]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact
import numpy as np
import warnings
import statistics as stt
#import bamboolib as bam
#from brokenaxes import brokenaxes
import os
#import plotly.express as px

#  1. load all .exp files

In [88]:
def dirFiles(fDir, ending):
    fNames = []
    for i in os.listdir(fDir):
        if i.endswith(ending):
            fNames.append(i)
    return fNames

dataFiles = dirFiles('data/2022-11-04 Boron measurements', 'exp') # Here you can now exchange 'csv' for 'exp'

In [100]:
def outlierCorrection(measurement, factorSD):
    element_signal = np.array(measurement)
    mean = np.mean(element_signal, axis=0)
    sd = np.std(element_signal, axis=0)
    return [x for x in measurement if (x > mean - factorSD * sd) or (x < mean + factorSD * sd)]


def data_convert(data):
    data_float = []
    fil =  (data.str.contains('L|IC|C|H') == True)
    endnum = data[fil].index[0]
    for i in data[0:endnum]:
        data_float.append(float(i))
    return data_float

# 2. Functions to calculate averages of one measurement point

# B11 background substact

In [102]:
def calculateSingleFile_B11(filename, bg_11B, factorSD):
#upload data:
    df = pd.read_csv('data/2022-11-04 Boron measurements/' + filename, sep='\t', header=22)  # Hi Jie, I already inserted the additional argument here
#calculate average signal and average backgroud:

    B11_data_float = data_convert(df['11B']) #conversion to float required, as numbers are stored as str
    fils = []
    filw = []
        
    for i in B11_data_float:
        if (i > bg_11B) == True:
            fils.append(i)
        else:
            filw.append(i)

    B_11_aver = np.mean(outlierCorrection(fils, factorSD))
    B_11_backg_aver = np.mean(outlierCorrection(filw, factorSD))
    res = B_11_aver - B_11_backg_aver
    return(res)

# B10 background substact for IC

In [103]:
def calculateSingleFile_B10_IC(filename, bg_10B, factorSD):
#upload data:
    df = pd.read_csv('data/2022-11-04 Boron measurements/' + filename, sep='\t', header=22)  # Hi Jie, I already inserted the additional argument here
#calculate average signal and average backgroud:

    B10_data_float = data_convert(df['10B']) #conversion to float required, as numbers are stored as str
    fils = []
    filw = []
        
    for i in B10_data_float:
        if (i > bg_10B) == True:
            fils.append(i)
        else:
            filw.append(i)

    B_10_aver = np.mean(outlierCorrection(fils, factorSD))
    B_10_backg_aver = np.mean(outlierCorrection(filw, factorSD))
    res = B_10_aver - B_10_backg_aver
    return(res)

# B10 background substact for FC

In [104]:
def calculateSingleFile_B10_FC(filename, bg_10B, factorSD):
#upload data:
    df = pd.read_csv('data/2022-11-04 Boron measurements/' + filename, sep='\t', header=22)  # Hi Jie, I already inserted the additional argument here
#calculate average signal and average backgroud:
    B10_data_float = data_convert(df['10B']) #conversion to float required, as numbers are stored as str
    B_interference_left = data_convert(df['9.9'])
    B_interference_right = data_convert(df['10.2'])
    
    B10_data = outlierCorrection(B10_data_float, 2)
    B_interference_left_res = outlierCorrection(B_interference_left, 2)
    B_interference_right_res = outlierCorrection(B_interference_right, 2)

    B10_interference = (B_interference_right_res - B_interference_left_res)*(10-9.9)/(10.2-9.9)
    
    fils = []
    filw = []    
    for i in B10_data:
        if (i > bg_10B) == True:
            fils.append(i)
        else:
            filw.append(i)
    B_10_backg_aver = np.mean(outlierCorrection(filw, factorSD))
    
    B_10_aver = np.mean(B10_data)
    
    res=B_10_aver-B_10_backg_aver-B10_interference
    return(res)

# 3. Calculate averages of all measured points

In [193]:
res = []
for i in dataFiles:
    aver = calculateSingleFile_B11(i, 0.005, 2)/calculateSingleFile_B10_IC(i, 0.0005, 2)
    Neptune_res = {'sequence': i, 'average':aver}
    res.append(Neptune_res)

/opt/anaconda3/lib/python3.8/site-packages/numpy/core/fromnumeric.py:3419: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/lib/python3.8/site-packages/numpy/core/_methods.py:188: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


# 4. connect laser and average value

In [348]:
df_Laser = pd.read_csv('data/Seq_log_20221104_191152.csv', header=0)
Laser_sequece = df_Laser[' Sequence Number'].dropna()
res_energy = []
res_spotsize = []
res_HV = []
dfN = pd.DataFrame(res)

for i in dfN['sequence']:
    Nep_index = float(i.split('_')[0])
    mapping = Laser_sequece[Laser_sequece==Nep_index].index[0]
    
    res_energy.append(df_Laser.loc[mapping, ' Laser Energy (mJ)'])
    res_spotsize.append(df_Laser.loc[mapping, ' Spot Size (um)'])
    res_HV.append(df_Laser.loc[mapping, ' Laser HV (kV)'])


In [350]:
results = {'Laser Energy (mJ)': res_energy, 
           'Spot Size (um)': res_spotsize, 'Laser HV (kV)': res_HV}

Laser_res= pd.DataFrame(results)
resss = pd.concat([Laser_res, dfN.reindex(Laser_res.index)], axis=1)

In [366]:
filA = resss['sequence'].str.contains('A')
filB = resss['sequence'].str.contains('B')
filC = resss['sequence'].str.contains('C')
filD = resss['sequence'].str.contains('D')
filU = resss['sequence'].str.contains('U')

resss_A = resss[filA]
resss_B = resss[filB]
resss_C = resss[filC]
resss_D = resss[filD]
resss_U = resss[filU]
resss_U

,Laser Energy (mJ),Spot Size (um),Laser HV (kV),sequence,average
0,64.641,160.0 x 160.0,23.179,210_U.exp,4.467466
2,65.173,108.0 x 108.0,23.260,111_U.exp,6.762157
3,65.136,60.0 x 60.0,23.308,049_U.exp,7.230501
4,64.663,28.0 x 28.0,23.344,420_U.exp,4.715343
5,64.855,40.0 x 40.0,23.368,348_U.exp,4.890240
...,...,...,...,...,...
428,64.800,160.0 x 160.0,23.321,221_U.exp,5.231250
429,65.158,160.0 x 160.0,23.233,198_U.exp,4.758936
430,65.091,160.0 x 160.0,23.316,259_U.exp,4.949789
431,64.523,60.0 x 60.0,23.269,045_U.exp,7.329706


### Some old code

In [ ]:
#check for outliers for signal:
    element_signal = np.array(fils)
    mean = np.mean(element_signal, axis=0)
    sd = np.std(element_signal, axis=0)
    final_list_signal = [x for x in fils if (x > mean - factorSD * sd)]
    final_list_signal = [x for x in final_list_signal if (x < mean + factorSD * sd)]
    
#check for outliers for background:
    element_backg = np.array(filw)
    mean = np.mean(element_backg, axis=0)
    sd = np.std(element_backg, axis=0)
    final_list_backg = [x for x in filw if (x > mean - factorSD * sd)]
    final_list_backg = [x for x in final_list_backg if (x < mean + factorSD * sd)]
    

def calculateSingleFile(filename, bg_10B, factorSD):
#upload data:
    df = pd.read_csv('data/2022-02-15 Boron measurements/' + filename, sep='\t', header=22)  # Hi Jie, I already inserted the additional argument here
#calculate average signal and average backgroud:
    endnum = df['10B'][df['10B'] == 'L3'].index[0]
#    B_10 = []
    fils = []
    filw = []
#    for i in df['10B'][0:endnum]:
#        B_10.append(float(i)) #conversion to float required, as numbers are stored as str
#    for i in B_10:
    for i in df['10B'][0:endnum]:
        meas = float(i)
        if (meas > bg_10B) == True:
            fils.append(meas)
        else:
            filw.append(meas)

#    outlier_corrected_signal =outlierCorrection(fils, factorSD)
#    outlier_corrected_backg =outlierCorrection(filw, factorSD)

#    B_10_aver = sum(outlier_corrected_signal )/len(outlier_corrected_signal )
#    B_10_backg_aver = sum(outlier_corrected_backg)/len(outlier_corrected_backg)

    B_10_aver = np.mean(outlierCorrection(fils, factorSD))
    B_10_backg_aver = np.mean(outlierCorrection(filw, factorSD))
    
    return(B_10_aver, B_10_backg_aver)